# LDA 

In [10]:
import numpy as np
import pandas as pd
class LDAClassifier:
    def __init__(self):
        self.classes=None
        self.means=None
        self.covariance=None
        self.covariance_inv=None
        self.priors=None
        
    def fit(self,X,y):
        X=np.asarray(X,dtype=float)
        y=np.asarray(y).ravel()
        
        self.classes=np.unique(y)
        
        N=X.shape[0]
        
        p=X.shape[1]
        
        self.means={}
        self.priors={}
        
        pooled_covariance=np.zeros((p,p))
        
        for class_label in self.classes:
            X_class=X[y==class_label]
            N_k=X_class.shape[0]
            mean_vector=np.mean(X_class,axis=0)
            self.means[class_label]=mean_vector
            self.priors[class_label]=N_k/N
            centered_data=X_class-mean_vector
            
            covariance_k=(np.dot(centered_data.T,centered_data)/
                          (N_k-1))
            pooled_covariance+=((N_k-1)*covariance_k)
            
        self.covariance=(pooled_covariance/(N_k))
            
        self.covariance_inv=np.linalg.inv(self.covariance)
        
        return self
    
    def predict(self,X):
        X=np.asarray(X,dtype=float)
        if X.ndim==1:
            X=X.reshape(1,-1)
        predictions=[]
        for x in X:
            discriminant_scores=[]
            for class_label in self.classes:
                mean_vector=self.means[class_label]
                prior=self.priors[class_label]
                
                term1=np.dot(
                    np.dot(x,self.covariance_inv),mean_vector
                )
                
                term2=-0.5*np.dot(
                    np.dot(
                        mean_vector,self.covariance_inv
                    ),mean_vector
                )
                
                term3=np.log(prior)
                
                score=term1+term2+term3
                discriminant_scores.append(score)
            best_index=np.argmax(discriminant_scores)
            predicted_class=self.classes[best_index]
            predictions.append(predicted_class)
        return np.array(predictions)

In [11]:

import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import OneHotEncoder

X_train = np.array([
    ["Sunny", "Hot", "High", "Weak"],
    ["Sunny", "Hot", "High", "Strong"],
    ["Overcast", "Hot", "High", "Weak"],
    ["Rain", "Mild", "High", "Weak"],
    ["Rain", "Cool", "Normal", "Weak"],
    ["Rain", "Cool", "Normal", "Strong"],
    ["Overcast", "Cool", "Normal", "Strong"],
    ["Sunny", "Mild", "High", "Weak"],
    ["Sunny", "Cool", "Normal", "Weak"],
    ["Rain", "Mild", "Normal", "Weak"],
    ["Sunny", "Mild", "Normal", "Strong"],
    ["Overcast", "Mild", "High", "Strong"],
    ["Overcast", "Hot", "Normal", "Weak"],
    ["Rain", "Mild", "High", "Strong"]
])

y_train = np.array([
    "No", "No", "Yes", "Yes", "Yes", "No", "Yes",
    "No", "Yes", "Yes", "Yes", "Yes", "Yes", "No"
])

# Convert categorical data into numerical data
encoder = OneHotEncoder(handle_unknown="ignore")

X_encoded = encoder.fit_transform(X_train).toarray()

# Split the dataset
x_train, x_test, y_train, y_test = train_test_split(
    X_encoded, y_train, test_size=0.3, random_state=0
)

# Create and train LDA model
model = LinearDiscriminantAnalysis()
model.fit(x_train, y_train)

# Test dataset
X_test = np.array([
    ["Sunny", "Cool", "High", "Strong"],
    ["Overcast", "Mild", "Normal", "Weak"],
    ["Rain", "Mild", "High", "Strong"]
])

# Encode test data using the same encoder
X_test_encoded = encoder.transform(X_test).toarray()

# Prediction
y_pred = model.predict(X_test_encoded)

print("Predicted values:", y_pred)

# Evaluation
y_test_pred = model.predict(x_test)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nAccuracy Score:")
print(accuracy_score(y_test, y_test_pred))


Predicted values: ['No' 'Yes' 'No']

Confusion Matrix:
[[0 0]
 [5 0]]

Accuracy Score:
0.0
